# Real-Phi benchmark on Google Colab (GPU runtime)

Runtime → Change runtime type → **GPU** (T4 or better). This notebook:
1. clones `QasimAhmad-khan/769503` at branch `claude/nice-bell-2pd7ys`,
2. serves the **pinned** `microsoft/Phi-4-mini-instruct` with vLLM, once in higher precision (reference) and once 4-bit
   (candidate, bitsandbytes) — one server at a time, as in production,
3. runs `check-phi` (four roles through one endpoint) and `bench-phi` (schema validity, latency, token usage from the
   model's own tokenizer, peak GPU memory, decision agreement on frozen candidate sets),
4. saves `colab_results.json` for you to commit back or paste into the chat.

It never trades and makes no profitability claim. Colab sessions time out, so the 24-hour soak cannot run here.
Set `PHI_REVISION` to the commit you want to pin (the Hugging Face model page → *Files and versions* → commit hash).

In [ ]:
PHI_REVISION = "REPLACE_WITH_PINNED_COMMIT"   # required: never use a moving revision
BRANCH = "claude/nice-bell-2pd7ys"
!nvidia-smi
!git clone -b $BRANCH https://github.com/QasimAhmad-khan/769503.git repo
%cd repo
!pip -q install -e '.[test]' vllm bitsandbytes

In [ ]:
import subprocess, time, json, os, urllib.request, torch
bf16 = torch.cuda.is_bf16_supported()
DTYPE = "bfloat16" if bf16 else "half"   # T4 has no bf16
def serve(extra, log):
    cmd = ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", "microsoft/Phi-4-mini-instruct",
           "--revision", PHI_REVISION, "--max-model-len", "4096", "--max-num-seqs", "1", "--port", "8000",
           "--gpu-memory-utilization", "0.85"] + extra
    p = subprocess.Popen(cmd, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    for _ in range(600):
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2); return p
        except Exception:
            time.sleep(2)
    raise RuntimeError(open(log).read()[-3000:])
def config_for(url):
    cfg = json.load(open("config/paper.json"))
    cfg["phi"].update(backend="openai_compatible", endpoint=url, model_revision=PHI_REVISION, runtime="vllm",
                      quantization="per-run")
    json.dump(cfg, open("config/paper.colab.json", "w"), indent=2)
URL = "http://127.0.0.1:8000/v1/chat/completions"
config_for(URL)

In [ ]:
# 1) reference precision
ref = serve(["--dtype", DTYPE], "ref.log")
!python -m cqc check-phi --config config/paper.colab.json | tee check_phi_reference.json
!python -m cqc replay --days 2 --start-day 20 --db dev_ledger.sqlite > /dev/null   # frozen candidate sets (synthetic)
!python -m cqc bench-phi --config config/paper.colab.json --endpoint reference=$URL --ledger dev_ledger.sqlite --out bench_ref
ref.terminate(); ref.wait(); time.sleep(10)

In [ ]:
# 2) 4-bit candidate (same revision)
q4 = serve(["--dtype", DTYPE, "--quantization", "bitsandbytes", "--load-format", "bitsandbytes"], "q4.log")
!python -m cqc check-phi --config config/paper.colab.json | tee check_phi_q4.json
!python -m cqc bench-phi --config config/paper.colab.json --endpoint candidate=$URL --ledger dev_ledger.sqlite --out bench_q4
q4.terminate(); q4.wait()

In [ ]:
# 3) decision agreement reference vs 4-bit on identical frozen candidate sets + summary
r = json.load(open("bench_ref/phi_bench.json")); q = json.load(open("bench_q4/phi_bench.json"))
a = list(r["endpoints"].values())[0]; b = list(q["endpoints"].values())[0]
summary = {"phi_revision": PHI_REVISION, "dtype": DTYPE, "gpu": torch.cuda.get_device_name(0),
           "reference": a, "q4": b,
           "note": "decision picks per set are in each phi_bench.json; compare them for agreement; synthetic candidate "
                   "sets only - not evidence of trading value"}
json.dump(summary, open("colab_results.json", "w"), indent=2, default=str)
print(json.dumps({k: summary[k] for k in ("phi_revision", "dtype", "gpu")}, indent=2))
print("schema validity ref:", a["schema_validity"], " q4:", b["schema_validity"])
print("GPU peak MiB ref/q4:", a["gpu_peak_mib"], b["gpu_peak_mib"])